# 第 25 节：A2A 协同与反馈循环 —— 让每份报告都"过审"再交付

> 前置课程：第 24 节（Plan-and-Execute + ReAct + SQLite 持久化）、第 23 节（技能池）、第 22 节（LLM + Function Calling + MCP）
> 对应项目：SmartBrief（`smartbrief/`）—— 在第 24 节 `backend/agents` + `backend/graph` 的基础上继续演进

> **本课不仅带你"走读代码"，而是带你经历这段代码被设计出来的过程。**
> 每读到一个设计，都先逼自己回答一个问题：**为什么这样写？**
> 为什么写完报告还要人审？为什么打回重写而不是直接改？为什么最多只能返工 2 次？
> 为什么"强制通过"而不是直接判失败？……答案都在下面的开发旅程里。


## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 课程导入：产品经理又来了 | — |
| **Part 2** | 环境准备与项目结构（第 25 节新增） | `agents/reviewer.py` / `rewriter.py` |
| **Part 3** | 质量问题：为什么一次成稿不靠谱 | — |
| **Part 4** | Reviewer 演进：审核标准 + 容错 + 放行哲学 | `agents/reviewer.py` |
| **Part 5** | Rewriter 演进：返工为什么必须"带着锚点" | `agents/rewriter.py` |
| **Part 6** | 条件循环：上限、强制通过与单一职责 | `graph/workflow.py` |
| **Part 7** | 状态与持久化：让每一轮审核可追溯 | `graph/state.py` + `db/` |
| **Part 8** | 工程容错：超时、重试与降级 | `agents/scheduler.py` + `mcp_client/` |
| **Part 9** | 端到端演示：模拟 LLM 完整跑一遍 | 多模块联合 |
| **Part 10** | 总结、课后练习与下节课预告 | — |

---

> 💡 **面向学员**：很多同学拿到别人写的项目，习惯"从第一行读到最后一个文件"。
> 代码是读完了，可一旦被问到"为什么这里要留 `review_history`？为什么打回不由
> Scheduler 决定？为什么最后要强制通过？"就答不上来。
>
> 本课沿用前面的学习法：**原始问题 → 直觉写法(v0) → 踩坑 → 重构优化 → 项目最终代码**。
> 先自己想想"如果是我会怎么写"，再对比项目代码，你就能同时得到两个答案：
> 代码**是怎么写的**，以及代码**为什么长成这样**。
> 建议按顺序逐 Cell 运行，观察每一次输出。


---
## Part 1：课程导入 —— 产品经理又来了
---

### 1.1 新需求

你是 SmartBrief 的开发工程师。第 24 节刚交付"输入主题 → 自动产出报告"，产品经理又来了：

> "现在报告是能自动写了，但**质量没保证**。你看这几次生成的结果：
> 有的全是'可能、大概'没有数据；有的数据是模型自己编的；还有的写到一半跑题了。
> 我要你加一道**质量关**：写完必须有人审，不合格就打回去重写，最多返工 2 次。
> 而且每次审核说了什么意见，都要能查得出来 —— 我们得对客户解释得清。"

> 🤔 **暂停思考**：如果让你来实现"审核 → 打回重写"这个闭环，你会怎么设计？
> 提示：这已经不是第 24 节的"一条直线跑到底"了，图里需要出现**分叉**和**回头路**。


In [1]:
# ============================================================
# Cell 0：v0 —— 回顾第 24 节交付的"直线流程"，缺口在哪？
# ============================================================
print("=" * 55)
print("  第 24 节的图：一条直线跑到底")
print("=" * 55)
print()
print("  START")
print("    |")
print("    v")
print("  planner   （拆计划）")
print("    |")
print("    v")
print("  scheduler（派发执行：researcher / writer）")
print("    |")
print("    v")
print("  END   <- 写完就交付，没有任何质量把关")
print()
print("  [问题1] 没人检查草稿：数据是不是编的？跑没跑题？字够不够？")
print("  [问题2] 就算发现有问题，也没有'回头路'可走")
print("  [问题3] 交付后说不清'质量为什么是这样'——没有审核记录")
print()
print("  产品经理要的，就是在这条直线上加一道'门'和一个'回路'：")
print("      scheduler -> reviewer --通过--> END")
print("                          |")
print("                          +--不通过--> rewriter --回到 reviewer-->")


  第 24 节的图：一条直线跑到底

  START
    |
    v
  planner   （拆计划）
    |
    v
  scheduler（派发执行：researcher / writer）
    |
    v
  END   <- 写完就交付，没有任何质量把关

  [问题1] 没人检查草稿：数据是不是编的？跑没跑题？字够不够？
  [问题2] 就算发现有问题，也没有'回头路'可走
  [问题3] 交付后说不清'质量为什么是这样'——没有审核记录

  产品经理要的，就是在这条直线上加一道'门'和一个'回路'：
      scheduler -> reviewer --通过--> END
                          |
                          +--不通过--> rewriter --回到 reviewer-->


### 1.2 拆解：真实团队怎么保证报告质量？

咨询公司出报告，永远有**写**和**审**两拨人：

| 角色 | 干什么 | 对应 Agent |
|------|--------|-----------|
| 写手 | 把研究数据写成报告章节 | Writer（第 24 节已有） |
| **质检员** | 拿着"硬标准"逐条核对，给出通过/不通过 + 具体意见 | **Reviewer（本节新增）** |
| **修改员** | 拿着审核意见**只改问题部分**，不动已通过的内容 | **Rewriter（本节新增）** |

让两个（甚至多个）Agent 像同事一样协作、互相审改，就叫做 **A2A（Agent-to-Agent）协同**。
第 24 节解决的是"**多智能体分工**"（Planner/Scheduler/Researcher/Writer 各司其职）；
本节解决的是"**多智能体互审与反馈**"——把"质量"这件事，从"碰运气"变成"有流程"。

> 生活类比：杂志社的"三审三校" ✍️
> 作者（Writer）交稿 → 编辑（Reviewer）审稿：数据有没有出处？有没有跑题？
> → 不合格退回修改（Rewriter）→ 再审 → 直到过审或主编拍板"就这样吧"（强制通过）。
>
> 这里有个关键**为什么**：为什么不能作者自己审自己的稿子？
> —— 作者对自己的文本有"思维惯性"，容易自我辩护、看不见盲点；
> 独立的 Reviewer 拿着**同一套客观标准**去审，才能避免"自嗨式交付"。
> 所以项目里 Reviewer 是**独立节点**，而不是在 Writer 内部加一句"请自查"。


### 1.3 本节课的"开发旅程"

```
质量线：一次成稿(碰运气)   -> 加Reviewer独立审核 -> 硬标准+容错+放行哲学
返工线：发现毛病没人改     -> Rewriter局部返工   -> 带主题锚点防跑题
控制线：直线图(没有回头路) -> 条件边(分叉)      -> 循环上限+强制通过
留痕线：只看最终结果       -> review_history多轮留痕 -> SQLite落库
工程线：LLM/网络不可靠     -> 超时/重试/降级    -> 流程永远能跑完
```


---
## Part 2：环境准备与项目结构
---

### 2.1 检查环境并把 backend 加进导入路径

本节直接复用第 24 节的运行环境（LangGraph + SQLite），新增的是 `reviewer.py` / `rewriter.py` 两个节点。
运行 Cell 1 后，后续 Cell 就能直接 `import` 项目源码了。


In [2]:
# ============================================================
# Cell 1：检查环境 + 把 backend 加入 sys.path
# ============================================================
import os
import sys

print("=" * 55)
print("  Python / LangGraph 环境检查")
print("=" * 55)
print(f"  Python 版本: {sys.version.split()[0]}")

# 兼容两种运行位置：
#   1) notebook 在 proj04 根目录运行时 -> backend = ./smartbrief/backend
#   2) notebook 在 smartbrief 目录运行时 -> backend = ./backend
BACKEND = os.path.join(os.getcwd(), "smartbrief", "backend")
if not os.path.isdir(BACKEND):
    BACKEND = os.path.join(os.getcwd(), "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)
print(f"  backend 路径: {BACKEND}")

try:
    import langgraph
    print(f"  [OK] langgraph 版本: {getattr(langgraph, '__version__', '>=1.0')}")
except ImportError as e:
    print("  [警告] 缺少 langgraph：", e)
    print("  请先安装：pip install 'langgraph>=1.0' 'langchain-core>=1.0'")


  Python / LangGraph 环境检查
  Python 版本: 3.10.20
  backend 路径: c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj04\smartbrief\backend
  [OK] langgraph 版本: >=1.0


### 2.2 项目结构一览（第 25 节新增部分）

```
smartbrief/backend/
├── agents/                      # 多智能体
│   ├── reviewer.py              #  ★ 新增：质量审核员（输出 passed + comments）
│   └── rewriter.py              #  ★ 新增：按意见返工（覆盖式修改草稿）
│   ├── planner.py / scheduler.py
│   └── react_researcher.py / react_writer.py   （第 24 节已有，基本不变）
├── graph/
│   ├── state.py                 #  ★ 白板新增第 5 组字段（审核相关）
│   └── workflow.py              #  ★ 加 reviewer/rewriter 节点 + 条件边
├── db/
│   ├── database.py              #  ★ tasks 表新增 6 列 + 旧库迁移
│   └── task_repository.py       #  ★ 新字段的序列化/白名单
├── mcp_client/mcp_client.py     #  ★ 工具调用超时 + 重试（第 25 节容错）
├── api/routes.py                #  /tasks/{id}/status 返回审核字段
└── main.py

frontend/src/views/TaskDetail.vue  # ★ 新增 A2A 审核卡片（分轮展示意见）
```

> 读代码前先记住一句话：**第 25 节所有改动，都是在回答一个"为什么"——**
> 为什么草稿不能直接交付？为什么打回不能无限次？为什么审核记录要留全？


---
## Part 3：质量问题 —— 为什么一次成稿不靠谱？
---

### 3.1 v0：让模型"自己写好一点"够不够？

产品经理要质量把关，最省事的方案是什么？—— 在 Writer 的 Prompt 里加一句"请写高质量报告"。
我们来拆穿这个方案的三个致命伤：

| # | 致命伤 | 具体表现 |
|---|--------|---------|
| 1 | **模型会"编数据"** | 没有搜到数据时，LLM 倾向于"合理地编"，而不是说"不知道" |
| 2 | **模型会"跑题"** | 草稿被截断/素材不足时，自由发挥出一篇和主题无关的报告 |
| 3 | **自己审自己 = 没审** | 同一模型既写又审，等于学生自己批改自己的卷子 |

> 🤔 **暂停思考**：质量保证的本质是什么？—— 是"**用独立于生产者的标准去校验产出物**"。
> Prompt 里的"请写好一点"只是**请求**，不是**校验**。所以必须有一个独立的 Reviewer，
> 拿着**可逐条判定**的硬标准，对草稿输出**结构化结论**（通过/不通过 + 意见）。


In [3]:
# ============================================================
# Cell 2：v0 —— "翻车现场"长什么样？
# ============================================================
print("=" * 55)
print("  一次成稿的三种'翻车现场'")
print("=" * 55)
print()

failures = [
    ("翻车1：编数据",
     "    据预测，智能手表市场未来几年将保持高速增长，市场规模大约在数百亿级别。\n"
     "    —— 通篇'可能/大概/约'，没有来源，数字无法核实"),
    ("翻车2：跑题",
     "    （草稿在超时后被截断成占位符）改写时模型失去主题锚点，\n"
     "    自由发挥出一篇'智能家居'相关的报告"),
    ("翻车3：无标准地自夸",
     "    Reviewer 提示词只写'请严格审核'，模型对每份稿子都说'质量很好，通过'"),
]
for title, detail in failures:
    print(f"  [{title}]")
    print(detail)
    print()

print("  >>> 结论：质量把关 = 独立 Reviewer + 硬标准 + 结构化输出")
print("  >>> 三个要素缺一个，这道'门'就是摆设")


  一次成稿的三种'翻车现场'

  [翻车1：编数据]
    据预测，智能手表市场未来几年将保持高速增长，市场规模大约在数百亿级别。
    —— 通篇'可能/大概/约'，没有来源，数字无法核实

  [翻车2：跑题]
    （草稿在超时后被截断成占位符）改写时模型失去主题锚点，
    自由发挥出一篇'智能家居'相关的报告

  [翻车3：无标准地自夸]
    Reviewer 提示词只写'请严格审核'，模型对每份稿子都说'质量很好，通过'

  >>> 结论：质量把关 = 独立 Reviewer + 硬标准 + 结构化输出
  >>> 三个要素缺一个，这道'门'就是摆设


---
## Part 4：Reviewer 演进 —— 标准、容错与"放行哲学"
---

### 4.1 审核标准从哪来？为什么是"硬标准"？

Reviewer 要审得让客户信服，标准必须**写在 Prompt 里、可逐条判定**。
项目里 `reviewer.py` 的审核标准是三条：

| 标准 | 通俗解释 | 为什么定这一条 |
|------|---------|---------------|
| ① **必须包含明确的数据点（≥2 处），如市场规模、增速、份额** | 报告不能通篇"大概/可能" | 客户要的是"干货"，不是形容词 |
| ② **每个数据点必须标注来源（如调研机构/新闻/工具结果）** | 数字要能追溯，不能是模型编的 | 反"幻觉"的第一道闸 |
| ③ **正文内容（不含标题）不少于 1000 字** | 内容要有足够的展开 | 防止"骨架稿"凑字数交付 |

> 为什么标准要设计成"**可判定**"而不是"写得优雅"这类**主观评价**？
>
> 1. **可解释**：驳回时能把意见落成"第②条：数据无来源"，客户听得懂；
> 2. **可稳定复现**：主观评分同一篇稿子两次结果可能不同，"硬标准"则基本一致；
> 3. **可执行**：Rewriter 能照着"哪一条没过"精准修改，而不是瞎猜"怎么改才叫优雅"。

另外注意一个工程细节：**审核意见条数要设上限**（项目里最多 5 条）。
为什么？—— 意见是给人/模型看的，一次甩 20 条谁也改不过来；而且 LLM 输出 JSON 太长更容易截断出错。


In [4]:
# ============================================================
# Cell 3：v1 —— 对比"没标准的审核"和"有标准的审核"
# ============================================================
print("=" * 55)
print("  v1 之前：Reviewer 提示词（无效版）")
print("=" * 55)
naive_prompt = '''你是一名报告审核员。请审核下面这份报告是否合格。
如果合格输出 {"passed": true}，不合格输出 {"passed": false, "comments": ["..."]}。

报告：
{report}'''
print(naive_prompt)
print()
print("  问题：什么叫'合格'？模型只能靠猜 —— 结果大概率对谁都放行")
print()
print("=" * 55)
print("  项目里的版本：三条硬标准，逐条可判定")
print("=" * 55)
criteria = '''一、报告必须包含明确的数据点（至少 2 处），如市场规模、增速、份额等；
二、每个数据点必须标注来源，禁止无来源的断言；
三、正文内容（不含标题）不得少于 1000 字。

请逐条核对，只输出 JSON：
通过 -> {"passed": true, "comments": []}
不通过 -> {"passed": false, "comments": ["第X条：具体问题"]}（最多 5 条）'''
print(criteria)


  v1 之前：Reviewer 提示词（无效版）
你是一名报告审核员。请审核下面这份报告是否合格。
如果合格输出 {"passed": true}，不合格输出 {"passed": false, "comments": ["..."]}。

报告：
{report}

  问题：什么叫'合格'？模型只能靠猜 —— 结果大概率对谁都放行

  项目里的版本：三条硬标准，逐条可判定
一、报告必须包含明确的数据点（至少 2 处），如市场规模、增速、份额等；
二、每个数据点必须标注来源，禁止无来源的断言；
三、正文内容（不含标题）不得少于 1000 字。

请逐条核对，只输出 JSON：
通过 -> {"passed": true, "comments": []}
不通过 -> {"passed": false, "comments": ["第X条：具体问题"]}（最多 5 条）


### 4.2 又一个老问题：LLM 的输出不保证是合法 JSON

第 24 节解析 Plan 时我们被 LLM"刁钻输出"坑过一次（包 Markdown 代码块、前后夹带废话）。
Reviewer 这里**一模一样**：审核结论必须解析成 `{"passed": ..., "comments": [...]}`，
而 LLM 可能输出：

- ```` ```json ... ``` ```` 代码块包裹；
- 前后多说了几句"我觉得……"；
- `comments` 字段偶尔不是数组。

所以和 `parse_plan` 对称，项目里也写了一个容错解析器 `parse_review()`。

### 4.3 "放行哲学"：解析失败时为什么放行而不是拦截？

这是一个容易想反的设计点。Reviewer 解析 JSON 失败时，项目**默认视为通过（放行）**：

```
解析失败 -> passed=True, comments=[]  （放行，记一条审核日志）
```

> 🤔 为什么"审核挂了"反而放行？这不等于没审吗？
>
> 看问题的角度要从"这一份稿子"跳到"整条产线"：
> - 主线目标是**产出报告并交付**，审核只是**质量门**；
> - 如果门坏了就卡死整条产线（任务失败），那是一次**功能故障**；
> - 放行 + 记录日志，则是**质量降级**——报告照常交付，问题被如实记录在案。
>
> 类比：质检仪坏了，工厂不会停产，而是"这批先放行、标记待复检"。
> 工程上这叫 **fail-open（故障时开放）**。而循环上限的"强制通过"（Part 6）就是给
> "放行"兜底的**最后一道保险**，保证任何情况下任务都能有输出、有交代。


In [5]:
# ============================================================
# Cell 4：v2 —— 手写容错解析器 parse_review（对应项目代码）
# ============================================================
import json
import re

MAX_COMMENTS = 5

def parse_review(content: str) -> dict:
    # 容错解析 Reviewer 输出：拿不到 JSON 就放行（fail-open）
    text = (content or "").strip()
    # 1. 去掉 ```json ... ``` 代码块
    text = re.sub(r"```(?:json)?\s*", "", text)
    # 2. 截取第一个 { 到最后一个 } 之间的内容
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        # 完全不是 JSON -> 放行（宁可漏审，不卡产线）
        return {"passed": True, "comments": []}
    try:
        data = json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return {"passed": True, "comments": []}
    # 3. 只认布尔 passed；comments 必须是列表且限制条数
    if not isinstance(data.get("passed"), bool):
        return {"passed": True, "comments": []}
    comments = data.get("comments")
    if not isinstance(comments, list):
        comments = []
    return {
        "passed": data["passed"],
        "comments": [str(c) for c in comments][:MAX_COMMENTS],
    }

# ---- 刁钻输出测试 ----
cases = [
    ('正常 JSON', '{"passed": false, "comments": ["第1条：数据点不足"]}'),
    ('Markdown 代码块包裹', '```json\n{"passed": false, "comments": ["第2条：无来源"]}\n```'),
    ('前后夹带废话', '我觉得这份报告还行吧。{"passed": true, "comments": []} 就这样。'),
    ('passed 是字符串(非法)', '{"passed": "yes", "comments": []}'),
    ('comments 不是数组(非法)', '{"passed": false, "comments": "差评"}'),
]
for name, raw in cases:
    print(f"  [{name}]")
    print(f"    输入: {raw[:60]}...")
    print(f"    解析: {parse_review(raw)}")


  [正常 JSON]
    输入: {"passed": false, "comments": ["第1条：数据点不足"]}...
    解析: {'passed': False, 'comments': ['第1条：数据点不足']}
  [Markdown 代码块包裹]
    输入: ```json
{"passed": false, "comments": ["第2条：无来源"]}
```...
    解析: {'passed': False, 'comments': ['第2条：无来源']}
  [前后夹带废话]
    输入: 我觉得这份报告还行吧。{"passed": true, "comments": []} 就这样。...
    解析: {'passed': True, 'comments': []}
  [passed 是字符串(非法)]
    输入: {"passed": "yes", "comments": []}...
    解析: {'passed': True, 'comments': []}
  [comments 不是数组(非法)]
    输入: {"passed": false, "comments": "差评"}...
    解析: {'passed': False, 'comments': []}


### 4.4 看看项目里的最终代码：reviewer.py

`reviewer_node()` 把上面所有设计串成 LangGraph 节点函数。它的职责一句话总结：
**读白板（草稿 + 当前轮次）→ 调 LLM 审核 → 把结论写回白板**（passed / comments /
iteration / review_history），并且**当达到循环上限时在此节点内完成"强制通过"**
（强制通过为什么放在这里，Part 6 会专门解释）。

Cell 5 直接打印项目源码走读（如果导入失败说明 backend 路径没配好，回 Cell 1）。


In [6]:
# ============================================================
# Cell 5：走读 agents/reviewer.py —— 最终代码
# ============================================================
import inspect

try:
    from agents import reviewer
    print("=" * 55)
    print("  parse_review 源码（4.2 的手写版 -> 项目版）")
    print("=" * 55)
    print(inspect.getsource(reviewer.parse_review))
except Exception as e:
    print("  [导入失败]", e)


  parse_review 源码（4.2 的手写版 -> 项目版）
def parse_review(content: str) -> dict:
    """容错解析 Reviewer 的 JSON 输出。

    容错链（与 Planner 的 parse_plan 同思路）：
    1. 剥掉 ```json ... ``` 代码块包裹
    2. 截取第一个 { 到最后一个 } 的 JSON 片段
    3. 仍失败 → 返回 passed=True（放行，不阻塞整个流程）
    """
    text = (content or "").strip()
    fence = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        text = text[start : end + 1]
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return {"passed": True, "comments": []}   # 解析失败：放行（不阻塞流程）
    return {
        "passed": bool(data.get("passed", True)),
        "comments": [str(c).strip() for c in data.get("comments", []) if str(c).strip()][:5],
    }



In [7]:
# ============================================================
# Cell 6：继续走读 —— reviewer_node 主逻辑
# ============================================================
try:
    from agents import reviewer
    print("=" * 55)
    print("  reviewer_node 源码")
    print("=" * 55)
    print(inspect.getsource(reviewer.reviewer_node))
    print()
    print("  重点关注：")
    print("   - 审核日志用 REVIEW_STEP_ID=0（不占 plan 的真实步骤）")
    print("   - 不通过时 iteration += 1")
    print("   - 达到 max_iterations 时：passed=True + forced_pass=True + 草稿追加警告")
    print("   - 每一轮 {round, passed, comments} 都追加进 review_history")
except Exception as e:
    print("  [导入失败]", e)


  reviewer_node 源码
async def reviewer_node(state: dict) -> dict:
    """LangGraph 节点：审核草稿 → 更新 passed / review_comments / iteration。

    强制通过逻辑（PRD 2.4）：
    - 审核通过 → passed=True，路由到 END
    - 审核不通过且 iteration < max_iterations → 打回（iteration += 1），路由到 Rewriter
    - 审核不通过且 iteration >= max_iterations → **强制通过**，草稿末尾追加警告标记

    多轮审核历史（review_history）：每次审核都追加一条 {"round", "passed", "comments"}，
    供任务详情页把每轮意见分轮展示，并与 ReAct 日志中的审核小节一一对应。
    """
    draft = state.get("draft_content") or ""
    logs = list(state.get("react_logs", []))
    iteration = int(state.get("iteration", 0))
    max_iterations = int(state.get("max_iterations", 3))
    history = list(state.get("review_history", []))

    round_no = iteration + 1          # 本轮审核编号（第 1 次审核时为 1）
    logs.append(
        {"type": "Thought", "content": f"开始审核草稿（第 {round_no} 次审核）", "step_id": REVIEW_STEP_ID}
    )
    logs.append(
        {"type": "Action", "content": "llm_gen(query=按 3 条标准评审报告草稿)", "step_id": REVIEW_STEP_ID}
    )
    try:

---
## Part 5：Rewriter 演进 —— 返工为什么必须"带着锚点"
---

### 5.1 谁来改？为什么是独立的 Rewriter？

草稿被驳回后，直觉做法是"把意见拼进原 Prompt，重新叫 Writer 写一遍"。
这有两个隐患：

| 做法 | 隐患 |
|------|------|
| 让 Writer 从头重写 | ① 全篇重写 = 已通过的好内容也可能被推翻 ② 原始研究数据早已不在 Writer 上下文里，越写越偏 |
| 让 LLM 做"局部补丁" | 让模型"改这几段、其余别动"要精确定位段落，LLM 做不到 → 补丁经常破坏原文结构 |

项目选的是第三条路：**独立的 Rewriter**。

### 5.2 覆盖式输出：与其"打补丁"，不如"带着锚点重写"

> 生活类比：文章返修不是让你在原文上贴便利贴，而是**把原文 + 审稿意见 + 原始素材**
> 一起交回给作者，让他"在原稿基础上改定稿"——改的是**内容**，不是**格式拼贴**。

Rewriter 拿到三样东西：
1. **上一版草稿**（已经写出的主体内容 → 好内容被保留的锚点）；
2. **审核意见 comments**（要改什么 → 返工清单）；
3. **主题 + 研究数据摘要**（不能跑题的锚点）。

输出是**覆盖整篇草稿的新 Markdown**（原稿 + 按意见修改后的全量重写）。

### 5.3 一个真实的 Bug 故事：为什么必须"主题锚定"

项目早期踩过一个坑：某次 Writer 因步骤超时被截断，落库的草稿只剩占位文本
"（步骤 3 撰写超时，内容待补充）"。Rewriter 的 Prompt 当时**没有携带主题**，
于是它拿着占位文本 + 一句"请修改"自由发挥，产出了一篇和用户主题完全无关的报告。

> 教训：**改稿的人必须知道"这篇稿子原本在写什么"**。
> 修复 = 双保险：
> ① 超时占位文本明确带上主题关键字（如 `f"（步骤 {id} 撰写超时，内容待补充）"`）；
> ② Rewriter 的 Prompt **强制携带 `topic` + 研究数据摘要**，让返工始终锚定主题。
>
> 这两个修复一个在 `scheduler.py`、一个在 `rewriter.py`，缺一不可。


In [8]:
# ============================================================
# Cell 7：v1 —— 演示"带锚点返工"的 Prompt 组装思路
# ============================================================
# 模拟一份"被截断"的草稿（对应 5.3 的真实 Bug 现场）
topic = "智能手表 2025 市场"
broken_draft = "（步骤 3 撰写超时，内容待补充）"
comments = ["第1条：数据点不足，需要市场规模/增速等具体数字并标注来源",
            "第2条：正文不足 1000 字"]

print("=" * 55)
print("  v0 危险写法：只给意见，不给主题")
print("=" * 55)
v0 = (
    "你是报告修改员，请根据意见修改草稿：\n"
    "意见：" + str(comments) + "\n"
    "草稿：" + broken_draft + "\n"
    "请输出修改后的完整报告："
)
print(v0)
print()
print("  >>> 模型看到占位草稿 + 无主题 -> 只能自由发挥 -> 跑题！")
print()
print("=" * 55)
print("  项目写法：主题 + 研究摘要 + 草稿 + 意见，四件套全带上")
print("=" * 55)

research_summary = (
    "市场规模：2025 年约 300 亿美元，同比增 15%（来源：示例调研）；"
    "主要厂商：Apple、华为、三星、小米（来源：示例搜索）"
)
anchored = (
    "你正在修改一份关于【" + topic + "】的市场调研报告。\n"
    "素材摘要（不得编造，可引用）：\n" + research_summary + "\n\n"
    "上一版草稿：\n" + broken_draft + "\n\n"
    "审核意见（逐条落实）：\n" + str(comments) + "\n\n"
    "请输出修改后的完整 Markdown 报告："
)
print(anchored)
print()
print("  >>> 注意：主题关键字出现在系统提示里 -> 模型跑题时会被'拉回来'")


  v0 危险写法：只给意见，不给主题
你是报告修改员，请根据意见修改草稿：
意见：['第1条：数据点不足，需要市场规模/增速等具体数字并标注来源', '第2条：正文不足 1000 字']
草稿：（步骤 3 撰写超时，内容待补充）
请输出修改后的完整报告：

  >>> 模型看到占位草稿 + 无主题 -> 只能自由发挥 -> 跑题！

  项目写法：主题 + 研究摘要 + 草稿 + 意见，四件套全带上
你正在修改一份关于【智能手表 2025 市场】的市场调研报告。
素材摘要（不得编造，可引用）：
市场规模：2025 年约 300 亿美元，同比增 15%（来源：示例调研）；主要厂商：Apple、华为、三星、小米（来源：示例搜索）

上一版草稿：
（步骤 3 撰写超时，内容待补充）

审核意见（逐条落实）：
['第1条：数据点不足，需要市场规模/增速等具体数字并标注来源', '第2条：正文不足 1000 字']

请输出修改后的完整 Markdown 报告：

  >>> 注意：主题关键字出现在系统提示里 -> 模型跑题时会被'拉回来'


### 5.4 看看项目里的最终代码：rewriter.py

`rewriter_node()` 和 `react_writer` 内部共用 `_summarize_research()` 把研究数据压成
Markdown 摘要，再连同 `topic`、上一版草稿、`comments` 一起发给 LLM。
如果 LLM 调用失败，则**保留原草稿**（宁可保持上一版，也不产出空稿）。


In [9]:
# ============================================================
# Cell 8：走读 agents/rewriter.py —— 最终代码
# ============================================================
import inspect

try:
    from agents import rewriter
    print("=" * 55)
    print("  rewriter_node 源码")
    print("=" * 55)
    print(inspect.getsource(rewriter.rewriter_node))
    print()
    print("  观察要点：")
    print("   - 输入里同时有 topic / research_data / draft_content / comments")
    print("   - 这就是 5.3 说的'防跑题锚点'")
    print("   - LLM 调用失败 -> 原样返回 draft_content（不制造空稿）")
except Exception as e:
    print("  [导入失败]", e)


  rewriter_node 源码
async def rewriter_node(state: dict) -> dict:
    """LangGraph 节点：按 review_comments 修改 draft_content（覆盖式）。

    :param state: AgentState（读 draft_content / review_comments / react_logs）
    :return: state 增量（draft_content 覆盖为修改稿 + react_logs 追加返工日志）
    """
    draft = state.get("draft_content") or ""
    comments = state.get("review_comments", [])
    topic = state.get("topic") or "（未知主题）"          # 研究主题（防止改写跑题的核心锚点）
    research_summary = _summarize_research(state.get("research_data") or {})   # 研究数据摘要
    logs = list(state.get("react_logs", []))

    logs.append(
        {"type": "Thought", "content": "开始根据审核意见修改草稿", "step_id": REWRITE_STEP_ID}
    )
    logs.append(
        {"type": "Action", "content": "llm_gen(query=按审核意见局部修改草稿)", "step_id": REWRITE_STEP_ID}
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"研究主题：{topic}\n\n"
                       f"可引用研究数据：\n{research_su

---
## Part 6：条件循环 —— 让"审核 → 返工"在图里转起来
---

### 6.1 为什么不能用"代码 while 循环"硬套？

有了 Reviewer 和 Rewriter 两个节点，直觉写法可能是：在某个节点里写 `while not passed: rewrite()`。
但项目没有这么做，而是把它**翻译成了图结构**——用 LangGraph 的**条件边**：

```
planner -> scheduler -> reviewer
                           |
         ┌─────────────────┘
         ▼ (passed=False 且未达上限)
      rewriter ──►（回到 reviewer，再来一轮）
         └─ (passed=True 或 达上限强制通过) ──► END
```

为什么"把循环画进图里"比"写在代码里"更好？

1. **状态管理交给框架**：节点的输入/输出由 LangGraph 自动合并进 `AgentState`，
   我们不用手写"上一轮结果放哪、怎么传下一轮"；
2. **看得见**：`app.get_graph()` 能直接画出"谁会回头、什么条件下回头"，
   出了 bug 一眼定位，客户/同事也好理解流程；
3. **为第 26 节铺路**：图结构天然支持"暂停 / 人工介入 / 断点续跑"（Checkpointer），
   而手写 while 循环做不到。

### 6.2 路由函数：条件边怎么知道往哪走？

LangGraph 的条件边需要一个"路由函数"：读白板，返回一个**字符串钥匙**：

```python
def route_after_review(state):
    # 只读白板，不改白板 —— 这是节点的活，不是边的活
    return "passed" if state.get("passed") else "rewrite"

graph.add_conditional_edges("reviewer", route_after_review, {
    "passed": END,
    "rewrite": "rewriter",
})
graph.add_edge("rewriter", "reviewer")   # 返工完，回到 reviewer 再来一轮
```

> 💡 注意路由函数和节点的分工：**边只做"看路牌"，节点才做"改状态"**。
> 这解释了后面一个问题：为什么"强制通过"发生在 reviewer 内部，而不是路由里——
> 因为**改状态**（iteration+1、追加警告、置 passed）是节点的职责；
> 路由函数永远只读不写，才能保持单一职责、好测试。


In [10]:
# ============================================================
# Cell 9：构建真实工作流，观察"条件边"长什么样
# ============================================================
try:
    from graph import workflow

    app = workflow.build_graph()
    g = app.get_graph()
    print("=" * 55)
    print("  [OK] LangGraph 编译成功！")
    print("=" * 55)
    print()
    print("  节点:", list(g.nodes.keys()))
    print()
    print("  边（source => target）：")
    for e in g.edges:
        print("   ", e.source, "=>", getattr(e, "target", None))
    print()
    print("  >> 对比第 24 节：现在多了 reviewer / rewriter 两个节点")
    print("  >> 以及一条'条件边'（reviewer 之后有两个出口）")
except Exception as e:
    import traceback
    traceback.print_exc()
    print("  [构建失败]", e)


  [OK] LangGraph 编译成功！

  节点: ['__start__', 'planner', 'scheduler', 'reviewer', 'rewriter', '__end__']

  边（source => target）：
    __start__ => planner
    planner => scheduler
    reviewer => __end__
    reviewer => rewriter
    rewriter => reviewer
    scheduler => reviewer

  >> 对比第 24 节：现在多了 reviewer / rewriter 两个节点
  >> 以及一条'条件边'（reviewer 之后有两个出口）


### 6.3 为什么必须设"循环上限"？

Reviewer 判不通过 → Rewriter 返工 → 再审……理论上可以永远转下去。必须设上限，三个原因：

1. **钱**：每一轮审核 + 返工都是两次 LLM 调用，成本随轮数线性上涨；
2. **时间与体验**：任务跑 10 分钟还没完，前端轮询一直转圈，用户早就走了；
3. **死循环风险**：LLM 是概率模型，可能永远"不满意"——没有上限就没有出口。

所以白板里专门有两个字段：`iteration`（已打回几次）和 `max_iterations`（上限）。

### 6.4 计数语义：为什么"最多 3 次" = "返工 2 次"？

项目默认 `max_iterations = 3`，语义要掰扯清楚：

| iteration（打回次数） | 含义 | 后续动作 |
|----------------------|------|---------|
| 0 → 1 | 第 1 次审核不通过 | 返工 1 次 |
| 1 → 2 | 第 2 次审核不通过 | 返工 2 次 |
| 2 → 3 | 第 3 次审核仍不通过 | **已到上限 → 强制通过，不再返工** |

也就是：**初审 + 返工 2 次**。计数发生在 reviewer 判"不通过"的那一刻（`iteration += 1`），
一旦 `iteration >= max_iterations`，当轮直接 `passed=True`，并给草稿追加一段警告标记：

```markdown
> ⚠️ 部分章节未通过校验（已强制输出）
```

### 6.5 为什么"强制通过"而不是直接判任务失败？

> 🤔 这是最容易纠结的设计：产品要质量，凭什么烂稿也放行？

答：**"有输出 + 显眼警告" 永远好过 "没输出"**。三个理由：

1. LLM 本质是概率模型，改 10 次也未必能过"硬标准"，无限返工是**用钱买不到的质量**；
2. 对用户而言，一份带 ⚠️ 的完整报告 > 一个"任务失败"的空页面——前者至少可读、可二次加工；
3. **警告标记让"降级"可见**：`forced_pass` 落库 + 草稿追加 ⚠️，前端能明确告诉用户
   "这份报告有部分未过校验，请人工复核"——质量降级不等于质量黑箱。

这样设计后，**流程永远有终点、永远有产出、降级永远有交代**。


In [11]:
# ============================================================
# Cell 10：模拟循环语义 —— Reviewer 连续 3 次驳回会怎样？
# ============================================================
import asyncio
from unittest.mock import patch, AsyncMock

class _FakeMessage:
    def __init__(self, content):
        self.content = content
        self.tool_calls = None

class _FakeChoice:
    def __init__(self, message):
        self.message = message

class _FakeResp:
    def __init__(self, text):
        # 项目代码统一访问 resp.choices[0].message.content
        self.choices = [_FakeChoice(_FakeMessage(text))]

# 用一个"每次都驳回"的假 LLM，驱动真实的 reviewer_node
async def _always_reject(messages, **kwargs):
    return _FakeResp('{"passed": false, "comments": ["第1条：数据点不足，请补充市场规模等数字并标注来源"]}')

async def _demo():
    from graph.state import AgentState
    from agents import reviewer

    # 初始白板：iteration=0，上限 3
    state = {
        "task_id": "demo_review",
        "topic": "智能手表 2025 市场",
        "iteration": 0,
        "max_iterations": 3,
        "review_history": [],
        "draft_content": "## 市场规模\n智能手表市场在增长。",
        "current_step_index": 1,
        "completed_steps": [1],
        "react_logs": [],
    }

    with patch.object(reviewer, "chat", AsyncMock(side_effect=_always_reject)):
        for round_no in range(1, 4):
            out = await reviewer.reviewer_node(dict(state))
            state.update({k: v for k, v in out.items() if k != "react_logs"})
            state["react_logs"] = out.get("react_logs", state["react_logs"])
            tag = "强制通过!" if out.get("forced_pass") else ("打回返工" if not out.get("passed") else "通过")
            print(f"  第{round_no}轮审核: passed={out.get('passed')} iteration={state['iteration']} -> {tag}")
            if out.get("forced_pass"):
                print()
                print("  --- 草稿末尾被追加的警告标记 ---")
                print(state["draft_content"].splitlines()[-1])
                break

    print()
    print("  review_history（每一轮都留痕）：")
    for h in state["review_history"]:
        print("   ", h)

# ---- 运行入口：兼容两种内核状态 ----
# Jupyter 内核可能已存在"运行中的事件循环"，直接 asyncio.run() 会抛
# RuntimeError: cannot be called from a running event loop。
# 兼容方案：主线程没有运行中循环 -> asyncio.run 直接跑；
#           已有运行中循环（Jupyter）-> 在独立线程里另开一个新事件循环执行。
def _run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        asyncio.run(coro)
        return
    import threading
    holder = {}
    def _worker():
        holder["result"] = asyncio.run(coro)
    t = threading.Thread(target=_worker)
    t.start()
    t.join()
    return holder.get("result")

_run_async(_demo())


  第1轮审核: passed=False iteration=1 -> 打回返工
  第2轮审核: passed=False iteration=2 -> 打回返工
  第3轮审核: passed=True iteration=3 -> 强制通过!

  --- 草稿末尾被追加的警告标记 ---
> ⚠️ 部分章节未通过校验（已强制输出）

  review_history（每一轮都留痕）：
    {'round': 1, 'passed': False, 'comments': ['第1条：数据点不足，请补充市场规模等数字并标注来源']}
    {'round': 2, 'passed': False, 'comments': ['第1条：数据点不足，请补充市场规模等数字并标注来源']}
    {'round': 3, 'passed': False, 'comments': ['第1条：数据点不足，请补充市场规模等数字并标注来源']}


### 6.6 看看项目里的最终代码：workflow.py

三个关键位置（Cell 11 / Cell 12 会逐段打印）：
1. **`route_after_review()`**：只读白板返回 `"passed"` / `"rewrite"`（边不改状态）；
2. **`build_graph()`**：`add_conditional_edges` 注册条件边 + `add_edge("rewriter", "reviewer")` 形成回路；
3. **`run_task()`**：图跑完后把 `iteration / max_iterations / passed / forced_pass /
   review_history` 一起落库 —— 前端 `/status` 接口就是从这里读到审核信息的。


In [12]:
# ============================================================
# Cell 11：走读 graph/workflow.py —— 条件边与落库
# ============================================================
import inspect

try:
    from graph import workflow
    print("=" * 55)
    print("  route_after_review 源码（条件路由）")
    print("=" * 55)
    print(inspect.getsource(workflow.route_after_review))
    print()
    print("=" * 55)
    print("  build_graph 源码")
    print("=" * 55)
    print(inspect.getsource(workflow.build_graph))
except Exception as e:
    print("  [导入失败]", e)


  route_after_review 源码（条件路由）
def route_after_review(state: dict) -> str:
    """Reviewer 之后的**条件边**路由：通过 → END；不通过 → Rewriter 返工。

    强制通过已在 reviewer_node 内部处理（不通过且达上限时返回 passed=True），
    所以这里只需要判断 passed 一个字段，逻辑保持单一。
    """
    return "passed" if state.get("passed") else "rewrite"


  build_graph 源码
def build_graph():
    """构建并编译 LangGraph（第 25 节：带 Reviewer/Rewriter 条件循环）。

    教学要点：
    - `add_conditional_edges("reviewer", 路由函数, 映射表)`：根据 state 动态决定下一步
    - 路由函数返回映射表里的 key（"passed"→END / "rewrite"→rewriter）
    - `add_edge("rewriter", "reviewer")`：返工后**循环回 Reviewer 再审核**（不重跑 Scheduler）
    """
    graph = StateGraph(AgentState)
    graph.add_node("planner", planner_node)          # 节点1：拆解计划
    graph.add_node("scheduler", scheduler_node)      # 节点2：按依赖派发执行
    graph.add_node("reviewer", reviewer_node)        # 节点3（25 节）：审核草稿
    graph.add_node("rewriter", rewriter_node)        # 节点4（25 节）：按意见返工
    graph.add_edge(START, "planner")
    graph.add_edge("planner", "scheduler")
    g

In [13]:
# ============================================================
# Cell 12：继续走读 —— run_task（图结束后的统一落库）
# ============================================================
import inspect

try:
    from graph import workflow
    print("=" * 55)
    print("  run_task 源码")
    print("=" * 55)
    print(inspect.getsource(workflow.run_task))
    print()
    print("  观察要点：")
    print("   - 一次 ainvoke 跑完整张图（含多轮审核循环）")
    print("   - 结束后统一 update_task：status + 五组审核字段一起写")
except Exception as e:
    print("  [导入失败]", e)


  run_task 源码
async def run_task(task_id: str, topic: str) -> dict:
    """执行完整流程：运行图 → 落库日志与最终状态。

    流程示意：
        1. 构造初始 state（白板）
        2. graph.ainvoke(initial)：LangGraph 自动按边执行
           planner（生成计划）→ scheduler（派发 ReAct 执行）
           → reviewer（审核草稿；A2A 不通过则返工给 rewriter 覆盖式重写，最多返工 2 次）
        3. 把图内产生的 react_logs 批量落库（task_logs 表）
        4. 把最终状态写回 tasks 表（status=completed），
           含 A2A 审核结果字段：review_comments / review_history / iteration /
           max_iterations / passed / forced_pass（前端详情页 A2A 卡片的数据源）
        5. 任一环节异常：把任务标记为 failed 并记录 error

    注意：任务初始记录（running）由 API 层调用 create_task 创建，
    本函数只负责执行与最终状态更新。

    :param task_id: 任务 id（由 API 层生成）
    :param topic: 用户提交的研究主题
    :return: 最终 AgentState（dict）
    """
    initial = _initial_state(task_id, topic)
    try:
        graph = build_graph()
        final = await graph.ainvoke(initial)          # 真正跑图（核心一行）
        final_report = final.get("draft_content") or "（报告草稿为空）"
        await _persist_logs(task_id

---
## Part 7：状态与持久化 —— 让每一轮审核都"可追溯"
---

### 7.1 白板（AgentState）新增第 5 组字段

第 24 节的白板字段分四组：任务信息 / Plan / ReAct 日志 / 报告数据。
第 25 节为了承载"审核循环"，加上了第 5 组 —— **A2A 审核字段**：

| 字段 | 含义 | 为什么需要 |
|------|------|-----------|
| `review_comments` | 最近一次审核的**意见列表** | Rewriter 返工的直接依据 |
| `review_history` | **每一轮**审核记录 `[{"round","passed","comments"}]` | 分轮可追溯（见下） |
| `iteration` | 已打回次数 | 循环计数 |
| `max_iterations` | 循环上限（默认 3） | 防死循环 / 控成本 |
| `passed` | 最近一次审核是否通过 | 条件边路由只读它 |
| `forced_pass` | 是否达上限被强制通过 | 前端显示 ⚠️、后端留证据 |

> 🤔 为什么有了 `review_comments` 还要 `review_history`？不是重复吗？
>
> `review_comments` 是**"当前动作需要"**的瞬态数据（给 Rewriter 用）；
> `review_history` 是**"事后审计需要"**的持久记录。两者的生命周期完全不同：
> - 客户问"这份报告为什么返工了两次？" → 翻 `review_history`，每轮意见清清楚楚；
> - 前端 A2A 卡片要"第 1 轮 → 意见 A / 第 2 轮 → 意见 B"分轮展示 → 只能靠 history；
> - 排查"为什么最后一版还是被强制通过" → history 里保留原始 passed=False 的判定，
>   而 `forced_pass` 只表示"最后放行了"。**原始判定不丢失**，这是审计的关键。


In [14]:
# ============================================================
# Cell 13：走读 graph/state.py —— 白板的第 5 组字段
# ============================================================
import inspect

try:
    from graph.state import AgentState
    print("=" * 55)
    print("  AgentState 最终定义（含第 25 节新增字段）")
    print("=" * 55)
    src = inspect.getsource(AgentState)
    print(src)
    print("  数一数：五组字段 —— 任务信息 / Plan / ReAct / 报告 / A2A 审核")
except Exception as e:
    print("  [导入失败]", e)


  AgentState 最终定义（含第 25 节新增字段）
class AgentState(TypedDict):
    """多智能体共享状态（"白板"）。

    所有 Agent 节点都读写这张"白板"：
    - 节点函数接收完整 state，返回**增量 dict**（只包含要修改的字段）
    - LangGraph 框架负责把增量合并回共享状态

    字段分五组：
    1. 任务信息：task_id / topic
    2. Plan 进度：plan / current_step_index / completed_steps
    3. ReAct 日志：react_logs
    4. 产出数据：research_data / draft_content / final_report / error
    5. A2A 审核循环（第 25 节）：review_comments / review_history / iteration / max_iterations / passed / forced_pass
    """
    # ---------- 1. 任务信息 ----------
    task_id: str              # 任务唯一 id（由 API 层生成，形如 task_ab12cd34）
    topic: str                # 用户提交的研究主题（如 "智能手表 2025 市场"）

    # ---------- 2. Plan 进度 ----------
    plan: List[Dict]                # 步骤列表：[{"id":1, "agent":"researcher", "task":"...", "depends_on":[]}]
                                    # agent 取值：researcher（联网研究）/ writer（撰写章节）
    current_step_index: int         # 当前进度下标（= 已完成步骤数，供前端 Plan 看板定位）
    completed_steps: List[int]      # 已完成步骤 id 

### 7.2 SQLite：老库怎么"加列"？—— ALTER TABLE 迁移

第 24 节的 `tasks` 表已经建好、里面甚至有历史数据了。新字段不能靠改 `CREATE TABLE` 解决，
因为 `CREATE TABLE IF NOT EXISTS` **只在新库生效**，对已存在的旧表什么都不做。

项目里的做法是两层配合：

1. **`_SCHEMA` 里写全新列**：全新环境直接建出完整表；
2. **`_MIGRATIONS` 列表逐个 `ALTER TABLE ADD COLUMN`**：旧库逐列补齐，
   列已存在时 SQLite 抛 `OperationalError` → 捕获跳过（**幂等**，可反复执行）。

```python
_MIGRATIONS = [
    "ALTER TABLE tasks ADD COLUMN review_comments TEXT",
    "ALTER TABLE tasks ADD COLUMN review_history TEXT",
    "ALTER TABLE tasks ADD COLUMN iteration INTEGER DEFAULT 0",
    "ALTER TABLE tasks ADD COLUMN max_iterations INTEGER DEFAULT 3",
    "ALTER TABLE tasks ADD COLUMN passed INTEGER",
    "ALTER TABLE tasks ADD COLUMN forced_pass INTEGER DEFAULT 0",
]
```

> 教学重点：**加一个新字段，不是改一行代码，而是五个地方要同步**：
> ① `_SCHEMA` 建表 SQL → ② `_MIGRATIONS` 迁移 → ③ `task_repository` 的 JSON 字段白名单
> → ④ `workflow.run_task` 落库 → ⑤ 前端展示。漏掉任何一处，轻则字段读不到，重则写库报错。
> （项目里把这条约定记在 CLAUDE.md 的第 12 条，回看代码注释也能找到。）


In [15]:
# ============================================================
# Cell 14：用临时库看真实表结构（不碰真实 data.db）
# ============================================================
import os
import sqlite3
import tempfile

from db import database

# 初始化一个"临时库"，隔离演示数据（init_db 支持传入自定义库路径）
tmp_dir = tempfile.mkdtemp()
tmp_db = os.path.join(tmp_dir, "lesson25.db")
print(f"  临时库: {tmp_db}")
database.init_db(tmp_db)          # 覆盖全局 DB_PATH 到临时库
print("  [1] init_db(临时库) 完成（幂等，重复执行不报错）")

conn = database.get_connection()
cols = [r["name"] for r in conn.execute("PRAGMA table_info(tasks)").fetchall()]
conn.close()
print()
print("  tasks 表全部列：")
print("   ", cols)
print()
print("  第 25 节新增的 6 列：", [c for c in cols if c in (
    "review_comments", "review_history", "iteration",
    "max_iterations", "passed", "forced_pass")])
print()
print("  教学要点：")
print("   - 全新库：_SCHEMA 直接建全列")
print("   - 旧库升级：靠 _MIGRATIONS 的 ALTER TABLE 补列（幂等）")


  临时库: C:\Users\13691\AppData\Local\Temp\tmp3b05d_kx\lesson25.db
  [1] init_db(临时库) 完成（幂等，重复执行不报错）

  tasks 表全部列：
    ['task_id', 'topic', 'status', 'plan', 'current_step_index', 'completed_steps', 'research_data', 'draft_content', 'final_report', 'error', 'review_comments', 'review_history', 'iteration', 'max_iterations', 'passed', 'forced_pass', 'created_at', 'updated_at']

  第 25 节新增的 6 列： ['review_comments', 'review_history', 'iteration', 'max_iterations', 'passed', 'forced_pass']

  教学要点：
   - 全新库：_SCHEMA 直接建全列
   - 旧库升级：靠 _MIGRATIONS 的 ALTER TABLE 补列（幂等）


In [16]:
# ============================================================
# Cell 15：走读 db —— 迁移列表 + update_task 落库
# ============================================================
import inspect

try:
    from db import database, task_repository

    print("=" * 55)
    print("  database._MIGRATIONS（旧库逐列迁移，幂等）")
    print("=" * 55)
    print(inspect.getsource(database.init_db))
    print("  _MIGRATIONS = ")
    for m in database._MIGRATIONS:
        print("   ", m)

    print()
    print("=" * 55)
    print("  task_repository.update_task（JSON 字段自动序列化）")
    print("=" * 55)
    print(inspect.getsource(task_repository.update_task))
except Exception as e:
    print("  [导入失败]", e)


  database._MIGRATIONS（旧库逐列迁移，幂等）
def init_db(db_path: Path | None = None) -> None:
    """初始化数据库：可选覆盖库文件路径（测试用临时库），并执行建表 SQL（幂等）。

    :param db_path: 指定库文件路径；不传则用默认 backend/data.db
                    （测试/notebook 演示可通过它隔离出临时库，避免污染真实数据）
    """
    global DB_PATH
    if db_path is not None:
        DB_PATH = Path(db_path)             # 覆盖全局库路径（模块级单例）
    with sqlite3.connect(DB_PATH) as conn:
        conn.executescript(_SCHEMA)         # 批量执行建表 SQL（IF NOT EXISTS → 幂等）
        for sql in _MIGRATIONS:             # 旧库补列（已存在则跳过）
            try:
                conn.execute(sql)
            except sqlite3.OperationalError:
                pass

  _MIGRATIONS = 
    ALTER TABLE tasks ADD COLUMN review_comments TEXT
    ALTER TABLE tasks ADD COLUMN review_history TEXT
    ALTER TABLE tasks ADD COLUMN iteration INTEGER DEFAULT 0
    ALTER TABLE tasks ADD COLUMN max_iterations INTEGER DEFAULT 3
    ALTER TABLE tasks ADD COLUMN passed INTEGER
    ALTER TABLE tasks ADD COLUMN forced_pass INTE

### 7.3 落库时机：为什么在"图跑完后"统一写一次？

第 24 节 `run_task()` 的持久化时机是**图执行完毕后统一落库**（`_persist_logs` + `update_task`），
第 25 节沿用了同样的设计，只是 update 时多了 6 个审核字段。为什么不在每轮审核后都写库？

1. **状态一致性**：`react_logs`、`review_history`、`iteration` 是一份"逻辑快照"，
   拆成多次写容易写一半崩溃，留下"日志有、状态没"的半成品；
2. **写库次数少**：每轮审核 2-3 次 LLM 调用 + 1 次写库 vs 每次都写库，IO 差一个量级；
3. **前端体验不依赖中间态**：任务状态是 `running → completed`，前端 2 秒轮询，
   完成后一次性拿到**完整的 Plan / 日志流 / 分轮审核记录**——比"边跑边漏"的中间态更可靠。

> 唯一例外：任务**失败**也要落库（`status=failed + error`），否则前端永远转圈。
> 这两条合起来就是 `run_task` 的 try/except 结构（见 Cell 12 源码）。


---
## Part 8：工程容错 —— 为什么第 25 节要集中处理"不可靠"？
---

### 8.1 外部依赖的"三宗罪"

把报告从"写完"升级到"过审"，意味着链路更长：Writer → Reviewer →（Rewrite → Reviewer）→ 落库。
链路越长，越容易断。三个最常见的断点：

| 断点 | 表现 |
|------|------|
| **慢** | 网络抖动、模型排队，一次调用可能卡几十秒 |
| **挂** | LLM 服务 5xx、MCP 工具失联 |
| **坏** | 返回了非 JSON / 空内容 / 乱码 |

项目的容错是**分层**做的：

| 层 | 手段 | 在哪个文件 | 效果 |
|----|------|-----------|------|
| 工具层 | MCP 调用 5 秒超时 + 失败重试 2 次 | `mcp_client/mcp_client.py` | 工具挂了自动换一次机会 |
| 步骤层 | 每个 Agent 步骤 `asyncio.wait_for` 限时 | `agents/scheduler.py` | 某一步卡死不拖垮全流程 |
| 节点层 | LLM 调用失败 → 放行 / 保留原稿 / 降级计划 | planner/reviewer/rewriter/writer | 节点失败不中断主线 |
| 任务层 | 整体异常 → `status=failed` + error 落库 | `graph/workflow.py` | 前端能明确看到失败原因 |

### 8.2 一个真实的 Bug：Writer 超时为什么是 120 秒而不是 30 秒？

这是第 25 节最值得讲的一个坑：

> 早期 Writer 和 Researcher 共用同一个步骤超时（约 30 秒）。Researcher 单步就是一次搜索，
> 30 秒绰绰有余；但 **Writer 要在一次调用里生成整篇章节/报告**，经常超过 30 秒。
> 结果：Writer 被 `asyncio.wait_for` 掐断 → 草稿退化成占位符
> `（步骤 3 撰写超时，内容待补充）` → 进入 A2A 返工时，Rewriter 面对一段占位文本，
> 没有主题约束就会"自由发挥" → 产出与用户主题无关的"臆想报告"。

> 修复分两步（都是"为什么"的答案）：
> 1. **分开设超时**：Researcher 30s / Writer 120s —— 因为两者单步工作量完全不同，
>    一刀切必然牺牲一方（太短掐断 Writer，太长拖死流程）；
> 2. **Rewriter 带主题锚点**（Part 5.3）—— 即使草稿被截断成占位符，
>    模型也始终知道"这篇报告原本要写什么"，不会跑题。

> 💡 一句话总结这一 Part：**代码要写"正常路径"是本能；把"异常路径"也设计好，
> 才是多智能体应用能上生产的门槛。**


In [17]:
# ============================================================
# Cell 16：走读容错设计 —— 超时常量 + 降级分支
# ============================================================
import inspect

try:
    from agents import scheduler
    print("=" * 55)
    print("  scheduler 超时常量（为什么是 30 / 120）")
    print("=" * 55)
    print("  AGENT_STEP_TIMEOUT_SECONDS  =", scheduler.AGENT_STEP_TIMEOUT_SECONDS)
    print("  WRITER_STEP_TIMEOUT_SECONDS =", scheduler.WRITER_STEP_TIMEOUT_SECONDS)
    print()
    src = inspect.getsource(scheduler.scheduler_node)
    # 只打印与超时降级相关的段落，避免整段源码刷屏
    lines = src.splitlines()
    for i, line in enumerate(lines):
        if "wait_for" in line or "TimeoutError" in line or "占位" in line or "超时" in line:
            lo = max(0, i - 1)
            hi = min(len(lines), i + 4)
            print("\n".join(lines[lo:hi]))
            print("    ...")
except Exception as e:
    print("  [导入失败]", e)


  scheduler 超时常量（为什么是 30 / 120）
  AGENT_STEP_TIMEOUT_SECONDS  = 30
  WRITER_STEP_TIMEOUT_SECONDS = 120

                # 研究步骤：ReAct 循环（Thought→Action(web_search)→Observation）
                # 容错：整个步骤限时，超时则跳过（记录降级结果，不卡死工作流）
                try:
                    result = await asyncio.wait_for(
                        react_researcher_run(step.get("task", ""), step_id, logs),
    ...
                try:
                    result = await asyncio.wait_for(
                        react_researcher_run(step.get("task", ""), step_id, logs),
                        timeout=AGENT_STEP_TIMEOUT_SECONDS,
                    )
    ...
                    research_data[f"step_{step_id}"] = result   # 研究结果按步骤号归档
                except asyncio.TimeoutError:
                    # 超时降级：写入空占位研究结果（observations=[] + skipped 标记），
                    # 流程不中断、不重试，依赖本步骤的 writer 会收到"数据缺失"信号
                    _append_log(
    ...
                except asyncio.TimeoutError:
                    # 超时降级：写入

In [18]:
# ============================================================
# Cell 17：MCP 工具层 —— 超时 + 重试（看 manager 的 _call_with_retry）
# ============================================================
try:
    from mcp_client import mcp_client
    print("=" * 55)
    print("  mcp_client 容错常量（模块级）")
    print("=" * 55)
    for name in ("MCP_CALL_TIMEOUT_SECONDS", "MCP_CALL_RETRIES", "CONNECT_TIMEOUT_SECONDS"):
        print(f"  {name} = {getattr(mcp_client, name, '?')}")
    print()
    print("  _call_with_retry 源码（失败自动重试 N 次再放弃）")
    print("=" * 55)
    import inspect
    fn = getattr(mcp_client.mcp_manager, "_call_with_retry", None)
    if fn is not None:
        print(inspect.getsource(fn))
except Exception as e:
    print("  [导入失败]", e)


  mcp_client 容错常量（模块级）
  MCP_CALL_TIMEOUT_SECONDS = 5.0
  MCP_CALL_RETRIES = 2
  CONNECT_TIMEOUT_SECONDS = 8

  _call_with_retry 源码（失败自动重试 N 次再放弃）
    async def _call_with_retry(self, conn, name: str, arguments: dict | None) -> dict:
        """带超时与重试的工具调用。

        容错策略（第 25 节）：每次调用限时 5s，失败/超时重试 2 次；
        全部失败才抛出最后一次异常（由上层 ReAct 循环的 Observation 记录后降级处理）。
        """
        last_exc: Exception | None = None
        for attempt in range(MCP_CALL_RETRIES + 1):
            try:
                return await asyncio.wait_for(
                    conn.call_tool(name, arguments), timeout=MCP_CALL_TIMEOUT_SECONDS
                )
            except Exception as exc:  # noqa: BLE001 - 重试兜底，必须捕获所有异常
                last_exc = exc
                if attempt < MCP_CALL_RETRIES:
                    logger.warning(
                        "MCP 工具 %s 调用失败（第 %d/%d 次）：%s，重试中",
                        name, attempt + 1, MCP_CALL_RETRIES, exc,
                    )
        raise last_exc



---
## Part 9：端到端演示 —— 完整跑一遍（模拟 LLM，无需 API Key）
---

把前面所有零件拼起来：**真实 LangGraph 图（含审核循环）+ 模拟的 LLM / 搜索工具 + 临时
SQLite 库**，完整走一遍"关键词 → 计划 → ReAct 研究 → 撰写 → 审核 → 打回 → 返工 →
再审核 → 强制通过 → 落库"，并观察 `review_history` 如何把每一轮意见留痕。

> 💡 这里用 `unittest.mock` 替换掉各模块的 `chat()` 和 MCP 的 `call_tool()`，
> 模拟 LLM 的行为，让我们在**不花钱、不联网**的情况下看到完整闭环。
> 其中 Reviewer 被设置成"每次都驳回"——故意逼出**循环上限 + 强制通过**这条最完整的路径。
> 真实运行时只需去掉 patch，换成真实的 `chat()` / MCP 工具即可（配好 `backend/.env`）。


In [19]:
# ============================================================
# Cell 18：端到端演示 —— 模拟 LLM 完整跑通 run_task
# ============================================================
import asyncio
import json
import os
import tempfile
from unittest.mock import AsyncMock, patch

print("=" * 55)
print("  准备：假 LLM / 假搜索 / 临时库")
print("=" * 55)

# ---- 1. 假消息结构（模拟 OpenAI 响应对象） ----
class _FakeMessage:
    def __init__(self, content):
        self.content = content
        self.tool_calls = None

class _FakeChoice:
    def __init__(self, message):
        self.message = message

class _FakeResp:
    def __init__(self, text):
        # 项目代码统一访问 resp.choices[0].message.content
        self.choices = [_FakeChoice(_FakeMessage(text))]

# ---- 2. Planner 返回的执行计划：两轮研究 + 一个撰写 ----
plan_text = json.dumps([
    {"id": 1, "agent": "researcher", "task": "搜索智能手表 2025 市场规模与增速", "depends_on": []},
    {"id": 2, "agent": "researcher", "task": "搜索智能手表头部厂商与竞争格局", "depends_on": [1]},
    {"id": 3, "agent": "writer", "task": "撰写《智能手表 2025 市场》报告", "depends_on": [1, 2]},
], ensure_ascii=False)

# ---- 3. 假搜索：按 query 关键词返回不同结果 ----
def fake_search(name, arguments):
    query = (arguments or {}).get("query", "")
    if "厂商" in query or "竞品" in query:
        content = "Apple、华为、三星、小米占据前四（来源：示例调研）"
    else:
        content = "2025 年全球智能手表市场规模约 300 亿美元，同比增长约 15%（来源：示例调研）"
    return {"content": content, "source": "mock"}

# ---- 4. 假 LLM：看"系统提示词"识别当前角色，返回对应结果 ----
async def fake_chat(messages, **kwargs):
    system = ""
    for m in messages:
        if m.get("role") == "system":
            system = m["content"]
            break
    if "规划器" in system:                     # Planner：给出一份 3 步计划
        return _FakeResp(plan_text)
    if "研究员" in system:                     # Researcher：先搜一次，再结束
        user_msgs = [m for m in messages if m.get("role") == "user"]
        if len(user_msgs) <= 1:
            return _FakeResp('{"action": "web_search", "query": "智能手表 2025 市场"}')
        return _FakeResp('{"action": "done"}')
    if "撰写员" in system:                     # Writer：生成章节草稿
        return _FakeResp(
            "## 市场规模与竞争格局\n\n"
            "2025 年全球智能手表市场规模约 300 亿美元，同比增长约 15%（来源：示例调研）。\n\n"
            "头部厂商方面，Apple、华为、三星、小米占据前四（来源：示例调研）。\n\n"
            "整体来看，智能手表市场仍处于高速增长期。"
        )
    if "审核员" in system:                     # Reviewer：故意每次都驳回 -> 触发返工与强制通过
        return _FakeResp(
            '{"passed": false, "comments": ["第1条：数据需补充年份对比来源", "第2条：正文需补充行业趋势分析"]}'
        )
    if "改写员" in system:                     # Rewriter：带着主题锚点返工
        return _FakeResp(
            "## 市场规模与竞争格局\n\n"
            "2025 年全球智能手表市场规模约 300 亿美元，同比增长约 15%（来源：示例调研）。\n\n"
            "头部厂商 Apple、华为、三星、小米合计占据前四，竞争格局稳定（来源：示例调研）。\n\n"
            "发展趋势：健康监测、独立通信、AI 语音助手成为三大差异化方向；建议后续补充近三年增长率对比数据。"
        )
    return _FakeResp("（未知角色，占位回复）")

print("   [OK] 假 LLM / 假搜索已就绪")
print()

async def _main():
    from agents import planner, react_researcher, react_writer, reviewer, rewriter
    from db import database, task_repository
    from graph import workflow
    from mcp_client.mcp_client import mcp_manager

    # 隔离到临时库（不污染真实 data.db）
    database.init_db(os.path.join(tempfile.mkdtemp(), "e2e_a2a.db"))

    task_id = "task_e2e_a2a"
    topic = "智能手表 2025 市场"
    task_repository.create_task(task_id, topic)          # API 层同样的"先落 running"

    # 把 5 个模块里的 chat 全部替换成假 LLM；把 MCP 搜索替换成假搜索
    modules = [planner, react_researcher, react_writer, reviewer, rewriter]
    patches = [patch.object(mod, "chat", AsyncMock(side_effect=fake_chat)) for mod in modules]
    patches.append(patch.object(mcp_manager, "call_tool", AsyncMock(side_effect=fake_search)))
    for p in patches:
        p.start()
    try:
        print(f"  [开始] 任务 {task_id}（主题：{topic}）...")
        await workflow.run_task(task_id, topic)          # 真实图 + 假 LLM
    finally:
        for p in patches:
            p.stop()

    # ---------- 从 SQLite 读回最终记录 ----------
    print()
    print("=" * 55)
    print("  执行完成！从 SQLite 读回任务记录")
    print("=" * 55)
    task = task_repository.get_task(task_id)
    print(f"  status         = {task['status']}")
    print(f"  passed         = {task['passed']}")
    print(f"  forced_pass    = {task['forced_pass']}   <- 是否达上限强制通过")
    print(f"  iteration      = {task['iteration']}     <- 实际打回次数")
    print(f"  max_iterations = {task['max_iterations']}")

    history = task["review_history"]
    if isinstance(history, str):
        history = json.loads(history)
    print()
    print("  review_history（每一轮审核都留痕）：")
    for h in history:
        print(f"    round {h['round']} | passed={h['passed']} | comments={h['comments']}")

    draft = task["draft_content"] or ""
    print()
    print("  草稿末尾 80 字（应能看到强制通过的警告标记）：")
    print("   ", draft[-80:].replace("\n", " "))

# ---- 运行入口：兼容两种内核状态 ----
# Jupyter 内核可能已存在"运行中的事件循环"，直接 asyncio.run() 会抛
# RuntimeError: cannot be called from a running event loop。
# 兼容方案：主线程没有运行中循环 -> asyncio.run 直接跑；
#           已有运行中循环（Jupyter）-> 在独立线程里另开一个新事件循环执行。
def _run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        asyncio.run(coro)
        return
    import threading
    holder = {}
    def _worker():
        holder["result"] = asyncio.run(coro)
    t = threading.Thread(target=_worker)
    t.start()
    t.join()
    return holder.get("result")

_run_async(_main())


  准备：假 LLM / 假搜索 / 临时库
   [OK] 假 LLM / 假搜索已就绪

  [开始] 任务 task_e2e_a2a（主题：智能手表 2025 市场）...

  执行完成！从 SQLite 读回任务记录
  status         = completed
  passed         = 1
  forced_pass    = 1   <- 是否达上限强制通过
  iteration      = 3     <- 实际打回次数
  max_iterations = 3

  review_history（每一轮审核都留痕）：
    round 1 | passed=False | comments=['第1条：数据需补充年份对比来源', '第2条：正文需补充行业趋势分析']
    round 2 | passed=False | comments=['第1条：数据需补充年份对比来源', '第2条：正文需补充行业趋势分析']
    round 3 | passed=False | comments=['第1条：数据需补充年份对比来源', '第2条：正文需补充行业趋势分析']

  草稿末尾 80 字（应能看到强制通过的警告标记）：
    示例调研）。  发展趋势：健康监测、独立通信、AI 语音助手成为三大差异化方向；建议后续补充近三年增长率对比数据。  > ⚠️ 部分章节未通过校验（已强制输出）


---

### 9.1 看懂上面的输出

你会在输出里看到一条完整的"打回流水账"：

```
round 1 | passed=False | 意见A        ← 第 1 次审核不通过 → 返工
round 2 | passed=False | 意见A        ← 第 2 次审核不通过 → 返工
round 3 | passed=False | 意见A        ← 第 3 次仍不通过 → 已达上限
forced_pass=True                       ← 强制通过：流程有终点、有产出、有警告
草稿末尾出现 "> ⚠️ 部分章节未通过校验（已强制输出）"
```

这就是把 6.4 / 6.5 的语义**跑成了真实数据**。前端 TaskDetail 的 A2A 卡片、
数据库里的 `review_history` 列，数据源都是这一条记录。

> 换一种玩法：把 `fake_chat` 里"审核员"分支的返回值改成
> `'{"passed": true, "comments": []}'`，再跑一遍 —— 你会看到审核一次就通过，
> 没有返工、没有强制通过。**同一张图，两种结局**，这就是条件循环的"图表达能力"。


---
## Part 10：课程总结
---

### 我们经历了什么？

| 演进阶段 | 遇到的问题 | 解决方案 |
|---------|-----------|---------|
| v0 | 一次成稿，数据靠编、无质量校验 | 独立的 Reviewer 按"硬标准"审核 |
| 标准 | 审核标准含糊，模型靠猜 | 三条可判定标准 + 结构化 JSON 输出 |
| 容错 | LLM 输出不是合法 JSON / 审核服务挂了 | `parse_review` 容错解析 + 解析失败**放行**（fail-open） |
| 返工 | 草稿被驳回没人改 / 全篇重写会丢内容 | 独立 Rewriter：草稿+意见+主题+数据摘要，覆盖式返工 |
| 防跑题 | 占位草稿 + 无主题 → "臆想报告" | Prompt 强制携带 topic + 研究摘要（双保险） |
| 控制 | 直线图没有"回头路" | LangGraph 条件边：passed→END / rewrite→Rewriter 循环 |
| 成本 | 无限返工 = 烧钱 + 死循环 | `iteration`/`max_iterations`，上限 3 = 返工 2 次 |
| 交付 | 返工到上限还是不过怎么办 | **强制通过**：有输出 + ⚠️ 警告 > 没输出 |
| 留痕 | 说不清"为什么改了两版" | `review_history` 每轮留痕 + SQLite 落库 |
| 工程 | LLM/网络慢、挂、坏 | 分层超时/重试/降级（MCP→步骤→节点→任务） |

### 关键设计原则

1. **独立校验**：写和审分离，质量保证 = 独立生产者 + 可判定标准；
2. **先想终点**：任何循环都要问"出口在哪"，用上限 + 强制通过保证流程必有终点、必有产出；
3. **降级可见**：放行、跳过、强制通过都要**留痕**（日志/标记/落库），降级 ≠ 黑箱；
4. **单一职责**：节点改状态、条件边只读不写，逻辑才可测、可解释；
5. **图优于手写循环**：条件边让"分叉/回头路"可视化，也为 Checkpointer、人工介入留了口子。

### 下节课预告

**第 26 节：LangGraph 记忆与 Checkpointer**
- `checkpoints.db`：把"图运行时状态"自动持久化，服务重启后任务可续跑
- 断点/恢复与人工介入（Human-in-the-loop）—— 审核不通过时"请人工确认"
- 多轮任务上下文记忆与并发任务的隔离

---

## 课后练习

### 练习一：你是架构师 —— 改循环上限

产品经理说："返工 2 次太少，改成 5 次试试。"

请先思考再动手：
1. 改哪几处？（提示：`workflow._initial_state` 的默认值；数据库已建行的 `max_iterations`
   是默认 3 —— 迁移只对新列生效，**历史行**怎么办？）
2. 上限设成 100 会带来什么新问题？（成本、等待时间、死循环风险）
3. 是否应该让上限变成**用户可选参数**？如果要，哪些地方要跟着改？

### 练习二：给 Reviewer 加一条审核标准

打开 `backend/agents/reviewer.py`，在 `SYSTEM_PROMPT` 里加第 4 条标准：
"报告必须包含『市场趋势/未来展望』小节"。

1. 修改 Prompt 后，用 Cell 10 的模拟方式（假 LLM 每次驳回 + 真 `reviewer_node`）跑一轮，
   观察 `review_history` 的变化；
2. 思考：加了这条后，`fake_chat` 的"撰写员/改写员"分支要产出什么内容才能通过审核？
   —— 这正好演示了 **Prompt 约定如何跨 Agent 传导**。

### 练习三：观察真实运行（需 API Key）

配置好 `backend/.env` 后：
1. 启动后端：`uvicorn main:app --reload --port 8002`；
2. 前端打开任务详情页提交主题（如"低空经济 2025"）；
3. 观察 Plan 看板、ReAct 日志流与 **A2A 审核卡片**：有没有返工？返工了几轮？
   每轮意见分别是什么？
4. 故意选一个很难写够 1000 字的主题，观察是否触发"强制通过 + ⚠️ 标记"。

---

## 参考资料

- 规范：`多智能体协作开发PRD（拆分）.md` —— 子文档（第 25 节：A2A 审核循环与工程容错）
- 代码：`backend/agents/reviewer.py` / `rewriter.py` —— 审核与返工节点
- 代码：`backend/graph/workflow.py` —— 条件边 + 循环上限 + 统一落库
- 代码：`backend/graph/state.py` —— AgentState 第 5 组审核字段
- 代码：`backend/db/database.py` / `task_repository.py` —— 老库加列迁移与 JSON 字段
- 代码：`backend/agents/scheduler.py` + `mcp_client/mcp_client.py` —— 超时/重试/降级
- 官方：LangGraph 条件边文档 https://langchain-ai.github.io/langgraph/how-tos/branching/

> 🎉 恭喜你完成了本节课的学习！
> 现在你不仅知道 A2A 审核循环的代码长什么样，更知道它**为什么长成这样**：
> 为什么审、审什么、打回几次、到顶怎么办、怎么留痕、挂了怎么办——每一行设计都有原因。
